In [3]:
from xgboost import XGBClassifier

In [4]:
import pandas as pd
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

URL = "https://archive.ics.uci.edu/static/public/" "222/bank+marketing.zip"
outer = ZipFile(BytesIO(urlopen(URL).read()))
inner = ZipFile(BytesIO(outer.read("bank-additional.zip")))
df = pd.read_csv(inner.open("bank-additional/bank-additional.csv"), sep=";")
print(df.shape)

(4119, 21)


In [5]:
model = XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42)

In [6]:
X = df.drop(columns=["duration", "y"])
y = df["y"]

X_encoded = pd.get_dummies(X, drop_first=True)
y_encoded = df["y"].map({"yes": 1, "no": 0})


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, stratify=y_encoded
)

In [7]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


def metrics(X_test, y_test, model):
    y_pred = model.predict(X_test)
    y_probs = model.predict_proba(X_test)[:, 1]

    result = {
        "Accuracy": round(accuracy_score(y_test, y_pred), 2),
        "Precision": round(precision_score(y_test, y_pred), 2),
        "Recall": round(recall_score(y_test, y_pred), 2),
        "F1 Score": round(f1_score(y_test, y_pred), 2),
        "ROC AUC": round(roc_auc_score(y_test, y_probs), 2),
    }

    return pd.DataFrame(result, index=[0])

In [8]:
model.fit(X_train, y_train)

metrics(X_test, y_test, model)

# 	Accuracy	Precision	Recall	F1 Score	ROC AUC
# 0	  0.9	     0.595	   0.278	 0.379	   0.753

,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,0.9,0.65,0.24,0.35,0.78


In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 70, 100, 200, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.5],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)
best_model = grid_search.best_estimator_

metrics(X_test, y_test, grid_search)

Best Parameters: {'learning_rate': 0.01, 'n_estimators': 100}
Best CV Score: 0.9019726858877085


,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,0.9,0.8,0.13,0.23,0.8


In [10]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    # "n_estimators": [50, 70, 100, 200, 500],
    # "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.5],
    "max_depth": [2, 4, 8, 9, 10],
    "min_child_weight": [1, 2, 3, 4, 5],
    "gamma": [0, 1],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="precision",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)
best_model = grid_search.best_estimator_

print(model.score(X_train, y_train))

metrics(X_test, y_test, grid_search)

Best Parameters: {'gamma': 1, 'max_depth': 4, 'min_child_weight': 4}
Best CV Score: 0.6359518523355535
0.9441578148710167


,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,0.9,0.69,0.22,0.34,0.81


In [11]:
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

print(model.score(X_train, y_train))
metrics(X_test, y_test, model)

0.9280728376327769


,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,0.9,0.63,0.24,0.35,0.79


In [14]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

y = y.map({"no": 0, "yes": 1})
X = pd.get_dummies(X, drop_first=True, dtype=int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=42,
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

metrics(X_test, y_test, model)

ValueError: Input y contains NaN.